# LeetCode 3042 — Count Prefix and Suffix Pairs I bằng Python

Link đề: https://leetcode.com/problems/count-prefix-and-suffix-pairs-i/ · Độ khó: **Easy** · Chủ đề: Array, String, Trie, String Matching

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def countPrefixSuffixPairs(self, words: List[str]) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm số cặp chỉ số `(i, j)` với `i < j` sao cho `words[i]` vừa là **tiền tố (prefix)** vừa là **hậu tố (suffix)** của `words[j]`.
2. Nắm các công cụ chuỗi của Python: **cắt lát (slicing)** `s[:k]`, `s[-k:]`, và hai hàm `startswith` / `endswith`; biết vòng lặp lồng nhau để duyệt mọi **cặp** `i < j`.
3. Đi từ cách **so sánh từng ký tự bằng tay** → **slicing** → **`startswith` + `endswith`** → **Trie với khóa là cặp ký tự (đầu, cuối)** (hướng đi cho phiên bản khó hơn của bài).
4. Biết vì sao thứ tự `i < j` quan trọng, vì sao `words[i]` **không được dài hơn** `words[j]`, và vì sao cặp `(i, j)` chỉ đếm **một lần**.
5. Tránh các lỗi kinh điển: dùng `in` thay cho prefix/suffix, đếm cả `(j, i)`, quên kiểm tra độ dài, nhầm `s[-k:]` với `s[k:]`.

## 2. Đề bài & ví dụ

Với hai chuỗi `str1` và `str2`, ta nói hàm `isPrefixAndSuffix(str1, str2)` trả về `True` nếu `str1` **vừa là tiền tố, vừa là hậu tố** của `str2`; ngược lại trả về `False`.

Ví dụ: `isPrefixAndSuffix("aba", "ababa")` là `True` vì `"aba"` là tiền tố của `"ababa"` (ba ký tự đầu `aba`) và cũng là hậu tố (ba ký tự cuối `aba`). Còn `isPrefixAndSuffix("abc", "abcd")` là `False` vì `"abc"` là tiền tố nhưng không là hậu tố.

Cho mảng chuỗi `words` (chỉ số từ `0`). Hãy trả về **số cặp chỉ số `(i, j)` thỏa `i < j` và `isPrefixAndSuffix(words[i], words[j])` là `True`**.

**Ví dụ 1**

- Input: `words = ["a", "aba", "ababa", "aa"]`
- Output: `4`
- Giải thích: các cặp thỏa là `(0, 1)`, `(0, 2)`, `(0, 3)` và `(1, 2)`. Ví dụ `(1, 2)`: `"aba"` vừa là tiền tố vừa là hậu tố của `"ababa"`.

**Ví dụ 2**

- Input: `words = ["pa", "papa", "ma", "mama"]`
- Output: `2`
- Giải thích: các cặp thỏa là `(0, 1)` (`"pa"` trong `"papa"`) và `(2, 3)` (`"ma"` trong `"mama"`).

**Ví dụ 3**

- Input: `words = ["abab", "ab"]`
- Output: `0`
- Giải thích: chỉ có cặp `(0, 1)`, nhưng `"abab"` dài hơn `"ab"` nên không thể là tiền tố/hậu tố của `"ab"`.

**Ràng buộc:** `1 <= words.length <= 50`; `1 <= words[i].length <= 10`; `words[i]` chỉ gồm chữ cái tiếng Anh thường.

In [1]:
from typing import List
import random

examples = [
    (["a", "aba", "ababa", "aa"], 4),
    (["pa", "papa", "ma", "mama"], 2),
    (["abab", "ab"], 0),
]
for words, expected in examples:
    print("words =", words, "-> kỳ vọng", expected)

words = ['a', 'aba', 'ababa', 'aa'] -> kỳ vọng 4
words = ['pa', 'papa', 'ma', 'mama'] -> kỳ vọng 2
words = ['abab', 'ab'] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Tiền tố và hậu tố là gì?

- **Tiền tố (prefix)** của một chuỗi là đoạn **bắt đầu từ đầu chuỗi**. Chuỗi `"ababa"` có các tiền tố: `"a"`, `"ab"`, `"aba"`, `"abab"`, `"ababa"`.
- **Hậu tố (suffix)** là đoạn **kết thúc ở cuối chuỗi**. Chuỗi `"ababa"` có các hậu tố: `"a"`, `"ba"`, `"aba"`, `"baba"`, `"ababa"`.

Chuỗi nào cũng là tiền tố và hậu tố **của chính nó**. Điều này giải thích vì sao `isPrefixAndSuffix("aa", "aa")` là `True` (nhưng bài chỉ xét cặp `i < j`, hai vị trí khác nhau).

In [2]:
s = "ababa"
prefixes = [s[:k] for k in range(1, len(s) + 1)]
suffixes = [s[len(s) - k:] for k in range(1, len(s) + 1)]
print("tiền tố:", prefixes)
print("hậu tố :", suffixes)
print("vừa là tiền tố vừa là hậu tố:", [p for p in prefixes if p in suffixes])

tiền tố: ['a', 'ab', 'aba', 'abab', 'ababa']
hậu tố : ['a', 'ba', 'aba', 'baba', 'ababa']
vừa là tiền tố vừa là hậu tố: ['a', 'aba', 'ababa']


### 3.2. Cắt lát (slicing): `s[:k]` và `s[-k:]`

- `s[:k]` lấy **k ký tự đầu**.
- `s[-k:]` lấy **k ký tự cuối** (chỉ số âm đếm từ cuối: `-1` là ký tự cuối cùng).
- `s[len(s) - k:]` cho cùng kết quả với `s[-k:]` khi `k >= 1`.

Cẩn thận: `s[-0:]` giống `s[0:]`, tức là **cả chuỗi** chứ không phải chuỗi rỗng. Trong bài này mọi chuỗi có độ dài `>= 1` nên `k >= 1`, nhưng bạn hãy nhớ cái bẫy này khi viết bài khác.

In [3]:
s = "ababa"
print(s[:3])       # aba  (3 ký tự đầu)
print(s[-3:])      # aba  (3 ký tự cuối)
print(s[len(s) - 3:])   # aba
print(s[:0] == "")      # True : k = 0 ở đầu cho chuỗi rỗng
print(s[-0:])           # ababa : cái bẫy! -0 == 0 nên lấy cả chuỗi
print(s[:10])           # ababa : cắt lát vượt biên không báo lỗi, chỉ lấy tới hết chuỗi

aba
aba
aba
True
ababa
ababa


### 3.3. `startswith` và `endswith`

`a.startswith(b)` trả `True` nếu chuỗi `a` **bắt đầu bằng** `b`; `a.endswith(b)` trả `True` nếu `a` **kết thúc bằng** `b`. Nếu `b` dài hơn `a` thì kết quả luôn là `False`, nên ta không cần tự kiểm tra độ dài. Đây chính là hai hàm "đọc là hiểu" cho đề bài này.

In [4]:
print("ababa".startswith("aba"), "ababa".endswith("aba"))     # True True
print("abcd".startswith("abc"), "abcd".endswith("abc"))       # True False
print("ab".startswith("abab"), "ab".endswith("abab"))         # False False : b dài hơn a

True True
True False
False False


### 3.4. Duyệt mọi cặp `i < j`

Để xét **mỗi cặp không thứ tự đúng một lần**, ta cho `i` chạy từ `0` tới `n - 1` và `j` chạy từ `i + 1` tới `n - 1`. Với `n` phần tử có `n * (n - 1) / 2` cặp. Cách viết thứ hai là dùng `itertools.combinations(range(n), 2)` (tạo các cặp tăng dần).

In [5]:
from itertools import combinations

n = 4
pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]
print(pairs)                                       # 6 cặp = 4*3/2
print(pairs == list(combinations(range(n), 2)))    # True

[(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
True


### 3.5. Tuple làm khóa của dict

Tuple `("a", "b")` là bất biến (immutable) nên **dùng được làm khóa dict** (còn list thì không). Ta sẽ dùng điều này ở Cách 4: khóa của nút Trie là cặp `(ký tự đầu, ký tự cuối)`.

In [6]:
d = {}
d[("a", "b")] = 1
d[("a", "b")] = d.get(("a", "b"), 0) + 1
print(d)                       # {('a', 'b'): 2}
print(("a", "c") in d)         # False

{('a', 'b'): 2}
False


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Lấy `words = ["a", "aba", "ababa", "aa"]`. Ta liệt kê **mọi cặp** `(i, j)` với `i < j` và hỏi: `words[i]` có là tiền tố **và** hậu tố của `words[j]` không?

| cặp | words[i] | words[j] | tiền tố? | hậu tố? | tính |
|---|---|---|---|---|---|
| (0, 1) | a | aba | có (`a`) | có (`a`) | +1 |
| (0, 2) | a | ababa | có | có | +1 |
| (0, 3) | a | aa | có | có | +1 |
| (1, 2) | aba | ababa | có (`aba`) | có (`aba`) | +1 |
| (1, 3) | aba | aa | `aba` dài hơn `aa` | không | 0 |
| (2, 3) | ababa | aa | dài hơn | không | 0 |

Tổng cộng `4`. Vì `n <= 50`, chỉ có tối đa `50 * 49 / 2 = 1225` cặp, và mỗi cặp so sánh tối đa `10` ký tự, nên làm đúng như bảng trên là hoàn toàn chạy kịp.

In [7]:
words = ["a", "aba", "ababa", "aa"]
n = len(words)
for i in range(n):
    for j in range(i + 1, n):
        a, b = words[i], words[j]
        ok = len(a) <= len(b) and b[:len(a)] == a and b[len(b) - len(a):] == a
        print(f"({i}, {j}) {a!r:8} vs {b!r:8} -> {ok}")

(0, 1) 'a'      vs 'aba'    -> True
(0, 2) 'a'      vs 'ababa'  -> True
(0, 3) 'a'      vs 'aa'     -> True
(1, 2) 'aba'    vs 'ababa'  -> True
(1, 3) 'aba'    vs 'aa'     -> False
(2, 3) 'ababa'  vs 'aa'     -> False


## 5. Cách 1 — Brute Force / baseline: so từng ký tự bằng tay

**Ý tưởng.** Viết hàm `is_prefix_and_suffix_manual(a, b)` không dùng công cụ có sẵn:

1. Nếu `len(a) > len(b)` thì `False` ngay (không thể là tiền tố).
2. **Tiền tố:** với mỗi `k` từ `0` tới `len(a) - 1`, `a[k]` phải bằng `b[k]`.
3. **Hậu tố:** `a` kết thúc ở cuối `b`, nên `a[k]` phải bằng `b[len(b) - len(a) + k]`.

Rồi duyệt mọi cặp `i < j` và đếm. Cách này dài dòng nhưng cho thấy `startswith`/`endswith` thực chất làm gì bên trong.

In [8]:
def is_prefix_and_suffix_manual(a: str, b: str) -> bool:
    if len(a) > len(b):                 # a dài hơn b thì không thể là tiền tố/hậu tố
        return False
    offset = len(b) - len(a)            # vị trí bắt đầu của hậu tố trong b
    for k in range(len(a)):
        if a[k] != b[k]:                # sai ở tiền tố
            return False
        if a[k] != b[offset + k]:       # sai ở hậu tố
            return False
    return True


def count_pairs_manual(words: List[str]) -> int:
    n = len(words)
    count = 0
    for i in range(n):
        for j in range(i + 1, n):
            if is_prefix_and_suffix_manual(words[i], words[j]):
                count += 1
    return count


for words, expected in examples:
    print(count_pairs_manual(words), "(kỳ vọng", expected, ")")

4 (kỳ vọng 4 )
2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


**Độ phức tạp (Cách 1).** Gọi `n` là số từ, `L` là độ dài lớn nhất của một từ. Có `O(n²)` cặp, mỗi cặp so tối đa `L` vị trí nên **Time `O(n² · L)`**, **Space `O(1)`** (chỉ vài biến đếm).

**Nút thắt:** code dài, dễ sai chỉ số `offset + k`. Python đã có sẵn `startswith`/`endswith`, ta nên dùng chúng.

## 6. Ý tưởng thuật toán / bước đột phá

Có hai hướng cải tiến:

1. **Rút gọn code** (không đổi thuật toán): thay vòng so sánh ký tự bằng slicing rồi bằng `startswith` + `endswith`. Với ràng buộc nhỏ của phiên bản I, đây là lời giải chuẩn.
2. **Đổi cách nhìn** (cho phiên bản lớn hơn, `n` tới `10^5`): `a` vừa là tiền tố vừa là hậu tố của `b` khi và chỉ khi **dãy cặp** `(a[0], a[-1]), (a[1], a[-2]), ..., (a[L-1], a[0])` bằng `L` cặp đầu của `(b[0], b[-1]), (b[1], b[-2]), ...`. Thật vậy, `a[k] = b[k]` (tiền tố) và `a[L-1-k] = b[m-1-k]` (hậu tố, với `m = len(b)`) gộp lại chính là "cặp thứ `k` của `a` bằng cặp thứ `k` của `b`". Vậy ta chỉ cần một **Trie** mà mỗi cạnh là một cặp ký tự.

Bước 2 này nằm ngoài yêu cầu của bài I nhưng rất đáng biết, vì nó là hướng đi của bài tiếp theo trong chuỗi (Count Prefix and Suffix Pairs II).

In [9]:
a, b = "aba", "ababa"
pairs_a = [(a[k], a[len(a) - 1 - k]) for k in range(len(a))]
pairs_b = [(b[k], b[len(b) - 1 - k]) for k in range(len(b))]
print("cặp của a:", pairs_a)
print("cặp của b:", pairs_b)
print("a là prefix+suffix của b?", pairs_b[:len(a)] == pairs_a)

cặp của a: [('a', 'a'), ('b', 'b'), ('a', 'a')]
cặp của b: [('a', 'a'), ('b', 'b'), ('a', 'a'), ('b', 'b'), ('a', 'a')]
a là prefix+suffix của b? True


## 7. Các cách tiếp theo

### Cách 2 — Slicing

**Giải thích từng đoạn.**

- `b[:len(a)]` là đoạn đầu của `b` dài bằng `a`; so với `a` để kiểm tra tiền tố.
- `b[-len(a):]` là đoạn cuối của `b` dài bằng `a`; so với `a` để kiểm tra hậu tố. (An toàn vì `len(a) >= 1`.)
- Nếu `a` dài hơn `b`, cả hai đoạn cắt ra chính là `b` (ngắn hơn `a`) nên so sánh với `a` luôn `False`, nhưng ta vẫn nên viết kiểm tra độ dài cho rõ ràng.

In [10]:
def count_pairs_slicing(words: List[str]) -> int:
    count = 0
    n = len(words)
    for i in range(n):
        a = words[i]
        for j in range(i + 1, n):
            b = words[j]
            if len(a) <= len(b) and b[:len(a)] == a and b[-len(a):] == a:
                count += 1
    return count


for words, expected in examples:
    print(count_pairs_slicing(words), "(kỳ vọng", expected, ")")

4 (kỳ vọng 4 )
2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


In [11]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionSlicing:
    def countPrefixSuffixPairs(self, words: List[str]) -> int:
        count = 0
        for i in range(len(words)):
            for j in range(i + 1, len(words)):
                a, b = words[i], words[j]
                if len(a) <= len(b) and b[:len(a)] == a and b[-len(a):] == a:
                    count += 1
        return count


print(SolutionSlicing().countPrefixSuffixPairs(["pa", "papa", "ma", "mama"]))   # 2

2


**Độ phức tạp Cách 2:** vẫn **Time `O(n² · L)`** (mỗi lần cắt lát và so sánh tốn `O(L)`), **Space `O(L)`** vì lát cắt tạo chuỗi mới tạm thời.

### Cách 3 — `startswith` + `endswith` (đẹp nhất, dùng để nộp)

**Giải thích.** `b.startswith(a) and b.endswith(a)` đã gồm cả việc xử lý khi `a` dài hơn `b`. Ta gộp luôn vòng lặp đếm bằng `sum(...)` trên biểu thức boolean (`True` tính là `1`).

In [12]:
def count_pairs_startswith(words: List[str]) -> int:
    n = len(words)
    return sum(
        words[j].startswith(words[i]) and words[j].endswith(words[i])
        for i in range(n)
        for j in range(i + 1, n)
    )


for words, expected in examples:
    print(count_pairs_startswith(words), "(kỳ vọng", expected, ")")

4 (kỳ vọng 4 )
2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


**Độ phức tạp Cách 3:** **Time `O(n² · L)`**, **Space `O(1)`** phụ. Với `n <= 50` và `L <= 10`, chỉ khoảng `1225 * 10` thao tác nhỏ, chạy tức thì.

### Cách 4 — Trie với khóa là cặp (đầu, cuối)

**Giải thích từng đoạn** (đi theo thứ tự `j = 0, 1, 2, ...`, nhờ đó mọi từ đã có trong Trie đều có chỉ số nhỏ hơn):

1. Mỗi nút Trie là một dict `children` (khóa là cặp ký tự) và một bộ đếm `end` = số từ **kết thúc đúng tại nút này**.
2. Với từ `b` hiện tại, đi theo các cặp `(b[k], b[m-1-k])` từ `k = 0`. Sau mỗi bước, nút hiện tại có `end` từ trước đó (đã chèn) có độ dài bằng độ sâu hiện tại và có dãy cặp trùng `k + 1` cặp đầu của `b`. Chúng chính là các `a` thỏa đề, nên cộng `end` vào đáp án.
3. Đi xong thì **chèn `b` vào Trie** (tăng `end` ở nút cuối, tạo nút nếu chưa có). Chèn **sau** khi đếm để không tự ghép `b` với chính nó.

In [13]:
class TrieNode:
    def __init__(self):
        self.children = {}      # khóa: (ký tự đầu, ký tự cuối) -> TrieNode
        self.end = 0            # số từ kết thúc ở nút này


def count_pairs_trie(words: List[str]) -> int:
    root = TrieNode()
    answer = 0
    for b in words:
        m = len(b)
        # 1) đếm: đi theo các cặp của b và cộng end ở mỗi nút đi qua
        node = root
        for k in range(m):
            key = (b[k], b[m - 1 - k])
            if key not in node.children:
                break                      # không còn từ nào đi tiếp được
            node = node.children[key]
            answer += node.end
        # 2) chèn b vào Trie
        node = root
        for k in range(m):
            key = (b[k], b[m - 1 - k])
            if key not in node.children:
                node.children[key] = TrieNode()
            node = node.children[key]
        node.end += 1
    return answer


for words, expected in examples:
    print(count_pairs_trie(words), "(kỳ vọng", expected, ")")

4 (kỳ vọng 4 )
2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


**Độ phức tạp Cách 4:** mỗi từ đi qua tối đa `L` nút hai lần (đếm + chèn) nên **Time `O(n · L)`** (tính dict là `O(1)` trung bình), **Space `O(n · L)`** cho Trie. Với `n <= 50` thì Cách 3 đã đủ; Trie chỉ thật sự cần khi `n` rất lớn.

## 8. TRACE từng bước

Ta trace Cách 3 trên ví dụ 1 để thấy từng cặp `(i, j)`, và trace Cách 4 (Trie) để thấy bộ đếm `end` thay đổi thế nào.

In [14]:
def trace_pairs(words: List[str]) -> int:
    count = 0
    n = len(words)
    print("words =", words)
    for i in range(n):
        for j in range(i + 1, n):
            a, b = words[i], words[j]
            pre = b.startswith(a)
            suf = b.endswith(a)
            ok = pre and suf
            if ok:
                count += 1
            print(f"(i={i}, j={j}) a={a!r} b={b!r} prefix={pre} suffix={suf} -> {'đếm' if ok else 'bỏ'}, count={count}")
    return count


assert trace_pairs(["a", "aba", "ababa", "aa"]) == 4

words = ['a', 'aba', 'ababa', 'aa']
(i=0, j=1) a='a' b='aba' prefix=True suffix=True -> đếm, count=1
(i=0, j=2) a='a' b='ababa' prefix=True suffix=True -> đếm, count=2
(i=0, j=3) a='a' b='aa' prefix=True suffix=True -> đếm, count=3
(i=1, j=2) a='aba' b='ababa' prefix=True suffix=True -> đếm, count=4
(i=1, j=3) a='aba' b='aa' prefix=False suffix=False -> bỏ, count=4
(i=2, j=3) a='ababa' b='aa' prefix=False suffix=False -> bỏ, count=4


In [15]:
def trace_trie(words: List[str]) -> int:
    root = TrieNode()
    answer = 0
    for idx, b in enumerate(words):
        m = len(b)
        node = root
        gained = []
        for k in range(m):
            key = (b[k], b[m - 1 - k])
            if key not in node.children:
                break
            node = node.children[key]
            answer += node.end
            gained.append((k + 1, node.end))
        node = root
        for k in range(m):
            key = (b[k], b[m - 1 - k])
            node = node.children.setdefault(key, TrieNode())
        node.end += 1
        print(f"j={idx} b={b!r}: (độ sâu, end) đã đi qua = {gained} -> answer={answer}; chèn xong, end ở nút cuối = {node.end}")
    return answer


assert trace_trie(["a", "aba", "ababa", "aa"]) == 4

j=0 b='a': (độ sâu, end) đã đi qua = [] -> answer=0; chèn xong, end ở nút cuối = 1
j=1 b='aba': (độ sâu, end) đã đi qua = [(1, 1)] -> answer=1; chèn xong, end ở nút cuối = 1
j=2 b='ababa': (độ sâu, end) đã đi qua = [(1, 1), (2, 0), (3, 1)] -> answer=3; chèn xong, end ở nút cuối = 1
j=3 b='aa': (độ sâu, end) đã đi qua = [(1, 1)] -> answer=4; chèn xong, end ở nút cuối = 1


Bảng trace Cách 3 (ví dụ 1):

| (i, j) | a | b | prefix | suffix | count |
|---|---|---|---|---|---|
| (0, 1) | a | aba | True | True | 1 |
| (0, 2) | a | ababa | True | True | 2 |
| (0, 3) | a | aa | True | True | 3 |
| (1, 2) | aba | ababa | True | True | 4 |
| (1, 3) | aba | aa | False | False | 4 |
| (2, 3) | ababa | aa | False | False | 4 |

Trong trace Trie: từ `"a"` có cặp `(a, a)`; từ `"aba"` có cặp `(a, a), (b, b), (a, a)` nên đi qua nút `(a,a)` có `end = 1` (từ `"a"`), cộng `1`. Từ `"ababa"` đi qua `(a,a)` (cộng `1` từ `"a"`) rồi `(b,b)` rồi `(a,a)` với `end = 1` (từ `"aba"`), cộng thêm `1`. Từ `"aa"` có hai cặp `(a,a),(a,a)`: nút đầu có `end = 1` cộng `1` (từ `"a"`), nút sau là `"aa"` chưa từng chèn nên `end = 0`. Tổng `1 + 2 + 1 = 4`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (`startswith` + `endswith`): ngắn, dễ đọc, dư sức qua giới hạn của phiên bản I.

In [16]:
from typing import List

class Solution:
    def countPrefixSuffixPairs(self, words: List[str]) -> int:
        n = len(words)
        count = 0
        for i in range(n):
            for j in range(i + 1, n):                 # chỉ xét cặp i < j, mỗi cặp đúng một lần
                if words[j].startswith(words[i]) and words[j].endswith(words[i]):
                    count += 1
        return count


sol = Solution()
for words, expected in examples:
    print(words, "->", sol.countPrefixSuffixPairs(words))

['a', 'aba', 'ababa', 'aa'] -> 4
['pa', 'papa', 'ma', 'mama'] -> 2
['abab', 'ab'] -> 0


## 10. Vì sao thuật toán đúng

- **Không sót, không trùng:** vòng lặp `i < j` liệt kê **mỗi cặp vị trí đúng một lần**; ta kiểm tra điều kiện đề cho từng cặp rồi đếm, nên đáp án chính là số cặp thỏa.
- **Điều kiện đúng bản chất:** `words[j].startswith(words[i])` đúng khi `words[i]` là đoạn đầu của `words[j]`; `endswith` đúng khi là đoạn cuối. Hai điều kiện cùng đúng chính là "vừa tiền tố vừa hậu tố". Khi `words[i]` dài hơn `words[j]`, cả hai đều `False`, nên ta không đếm nhầm.
- **Hai chuỗi trùng nhau vẫn là một cặp hợp lệ:** `["a", "a"]` cho `1` (mỗi chuỗi là tiền tố, hậu tố của chính nó), đúng đề.
- **Cách Trie** đúng nhờ tương đương: `a` là prefix+suffix của `b` ⇔ `L` cặp `(a[k], a[L-1-k])` bằng `L` cặp đầu `(b[k], b[m-1-k])`.

## 11. Phân tích Time / Space Complexity

Gọi `n` là số từ, `L` là độ dài tối đa của một từ.

| Cách | Time | Space |
|---|---|---|
| 1. So ký tự bằng tay | `O(n² · L)` | `O(1)` |
| 2. Slicing | `O(n² · L)` | `O(L)` |
| 3. `startswith` / `endswith` | `O(n² · L)` | `O(1)` |
| 4. Trie cặp ký tự | `O(n · L)` | `O(n · L)` |

Với `n <= 50`, `L <= 10`: tối đa `1225` cặp, mỗi cặp vài chục thao tác, tức chỉ vài chục nghìn phép so sánh. Rất nhanh.

## 12. Bộ test

Ta gom các cách vào một dict `impls` và so sánh với kỳ vọng.

In [17]:
sol = Solution()
impls = {
    "manual": count_pairs_manual,
    "slicing": count_pairs_slicing,
    "startswith": count_pairs_startswith,
    "trie": count_pairs_trie,
    "leetcode": sol.countPrefixSuffixPairs,
}


def check_all(words, expected, label):
    for key, fn in impls.items():
        actual = fn(list(words))
        assert actual == expected, (label, key, words, actual, expected)
    print("PASS", label)


for idx, (words, expected) in enumerate(examples, 1):
    check_all(words, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3


### 12.1. Các trường hợp biên

In [18]:
check_all(["a"], 0, "một từ duy nhất")
check_all(["a", "a"], 1, "hai từ giống nhau")
check_all(["a", "a", "a"], 3, "ba từ giống nhau: C(3,2) = 3 cặp")
check_all(["ab", "ba"], 0, "cùng độ dài, khác nhau")
check_all(["abc", "ab"], 0, "từ trước dài hơn từ sau")
check_all(["ab", "abab"], 1, "ab trong abab")
check_all(["ab", "aab"], 0, "chỉ là hậu tố, không là tiền tố")
check_all(["a", "ab"], 0, "chỉ là tiền tố, không là hậu tố")
check_all(["a", "bab"], 0, "a nằm ở giữa, không phải đầu/cuối")
check_all(["a" * 10] * 50, 50 * 49 // 2, "50 từ aaaaaaaaaa: mọi cặp đều thỏa")

PASS một từ duy nhất
PASS hai từ giống nhau
PASS ba từ giống nhau: C(3,2) = 3 cặp
PASS cùng độ dài, khác nhau
PASS từ trước dài hơn từ sau
PASS ab trong abab
PASS chỉ là hậu tố, không là tiền tố
PASS chỉ là tiền tố, không là hậu tố
PASS a nằm ở giữa, không phải đầu/cuối
PASS 50 từ aaaaaaaaaa: mọi cặp đều thỏa


### 12.2. Phản ví dụ cho các cách làm sai

Hai "ý tưởng gần đúng" hay gặp: (a) chỉ kiểm tra `a in b` (xuất hiện ở đâu đó), (b) chỉ kiểm tra tiền tố. Ta xây hàm sai rồi tìm input làm lộ lỗi.

In [19]:
def wrong_in(words):
    n = len(words)
    return sum(words[i] in words[j] for i in range(n) for j in range(i + 1, n))


def wrong_prefix_only(words):
    n = len(words)
    return sum(words[j].startswith(words[i]) for i in range(n) for j in range(i + 1, n))


def wrong_both_directions(words):
    n = len(words)
    return sum(
        words[j].startswith(words[i]) and words[j].endswith(words[i])
        for i in range(n) for j in range(n) if i != j       # đếm cả (j, i)
    )


bad_in = ["a", "bab"]            # 'a' nằm ở giữa
bad_prefix = ["a", "ab"]         # chỉ là tiền tố
bad_dir = ["a", "a"]             # chỉ có i<j mới được tính: đáp án đúng là 1
print(wrong_in(bad_in), "!=", count_pairs_startswith(bad_in))
print(wrong_prefix_only(bad_prefix), "!=", count_pairs_startswith(bad_prefix))
print(wrong_both_directions(bad_dir), "!=", count_pairs_startswith(bad_dir))
assert wrong_in(bad_in) != count_pairs_startswith(bad_in)
assert wrong_prefix_only(bad_prefix) != count_pairs_startswith(bad_prefix)
assert wrong_both_directions(["a", "a"]) != count_pairs_startswith(["a", "a"])

1 != 0
1 != 0
2 != 1


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Dùng bảng chữ cái nhỏ (`a`, `b`) để các từ dễ trùng và lồng nhau, và tăng độ dài tới giới hạn đề. Mọi cách phải cho cùng kết quả.

In [20]:
def random_words(rng, alphabet="ab", max_len=10, max_n=50):
    n = rng.randint(1, max_n)
    return ["".join(rng.choice(alphabet) for _ in range(rng.randint(1, max_len))) for _ in range(n)]


rng = random.Random(3042)
for trial in range(1500):
    alphabet = rng.choice(["a", "ab", "abc", "abcdefghijklmnopqrstuvwxyz"])
    max_len = rng.choice([1, 2, 3, 5, 10])
    words = random_words(rng, alphabet, max_len)
    assert all(1 <= len(w) <= 10 and w.islower() for w in words) and 1 <= len(words) <= 50
    ref = count_pairs_manual(words)
    for key, fn in impls.items():
        assert fn(list(words)) == ref, (key, words)
print("PASS 1500 input ngẫu nhiên: mọi cách (kể cả Trie) khớp nhau")

PASS 1500 input ngẫu nhiên: mọi cách (kể cả Trie) khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `in` thay cho tiền tố/hậu tố

In [21]:
def sai_dung_in(words):
    count = 0
    for i in range(len(words)):
        for j in range(i + 1, len(words)):
            if words[i] in words[j]:           # SAI: 'a' in 'bab' là True nhưng a không ở đầu/cuối
                count += 1
    return count


print("sai:", sai_dung_in(["a", "bab"]), "| đúng:", Solution().countPrefixSuffixPairs(["a", "bab"]))

sai: 1 | đúng: 0


**Vì sao sai:** toán tử `in` hỏi "có xuất hiện **ở đâu đó** không", còn đề yêu cầu xuất hiện **ở đầu và ở cuối**. **Cách sửa:** dùng `startswith` và `endswith` (code đúng ở §9).

### Lỗi 2: Đếm cả hai chiều `(i, j)` và `(j, i)`

In [22]:
def sai_hai_chieu(words):
    count = 0
    for i in range(len(words)):
        for j in range(len(words)):          # SAI: j chạy hết, kể cả j < i và j == i
            if words[j].startswith(words[i]) and words[j].endswith(words[i]):
                count += 1
    return count


print("sai:", sai_hai_chieu(["a", "a"]), "| đúng:", Solution().countPrefixSuffixPairs(["a", "a"]))

sai: 4 | đúng: 1


**Vì sao sai:** với `j == i` mỗi từ luôn là prefix+suffix của chính nó (cộng thừa), và với `j < i` ta tính cả cặp bị đảo thứ tự. Đề chỉ đếm `i < j`. **Cách sửa:** `for j in range(i + 1, n)`.

### Lỗi 3: Nhầm `s[-k:]` với `s[k:]` (hoặc `s[:-k]`)

In [23]:
def sai_slicing(words):
    count = 0
    for i in range(len(words)):
        a = words[i]
        for j in range(i + 1, len(words)):
            b = words[j]
            if b[:len(a)] == a and b[len(a):] == a:      # SAI: b[len(a):] là phần SAU len(a) ký tự đầu
                count += 1
    return count


print("sai:", sai_slicing(["a", "aba"]), "| đúng:", Solution().countPrefixSuffixPairs(["a", "aba"]))

sai: 0 | đúng: 1


**Vì sao sai:** `b[len(a):]` bỏ `len(a)` ký tự đầu, giữ lại **phần còn lại**, không phải `len(a)` ký tự cuối. Với `a = "a"`, `b = "aba"`: `b[1:] = "ba"` khác `"a"`. **Cách sửa:** `b[-len(a):]` (hoặc `b[len(b) - len(a):]`).

### Lỗi 4: Quên chặn trường hợp `a` dài hơn `b` khi tự viết vòng ký tự

In [24]:
def sai_thieu_do_dai(a, b):
    for k in range(len(a)):
        if a[k] != b[k]:            # SAI: nếu a dài hơn b thì b[k] vượt biên
            return False
    return True


try:
    sai_thieu_do_dai("abc", "ab")
except IndexError as e:
    print("IndexError:", e)
print("đúng:", is_prefix_and_suffix_manual("abc", "ab"))

IndexError: string index out of range
đúng: False


**Vì sao sai:** chỉ số `k` vượt quá độ dài của `b` gây `IndexError`. **Cách sửa:** kiểm tra `len(a) > len(b)` trả `False` trước (như `is_prefix_and_suffix_manual`), hoặc dùng `startswith` vốn đã xử lý sẵn.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "tiền tố **và** hậu tố" → nghĩ `startswith` + `endswith`. Thấy "chỉ cần xuất hiện" mới nghĩ `in`.
- Thấy "cặp `i < j`" với `n` nhỏ (`<= 50`) → hai vòng lặp lồng là đủ; thấy `n` cỡ `10^5` → phải đổi cấu trúc dữ liệu (Trie, hash).
- Mẹo **Trie cặp**: biến điều kiện "đầu và cuối cùng khớp" thành **một** dãy khóa `(b[k], b[m-1-k])`, để dùng Trie thường (một chiều).
- Cộng `True` như `1`: `sum(điều_kiện for ...)` là cách đếm gọn trong Python.
- Luôn thử input `["a", "a"]` và `["ab", "a"]` để kiểm `i < j` và kiểm độ dài.

## 15. Biến thể của bài

1. **Trả về danh sách các cặp** `(i, j)` thay vì chỉ đếm.
2. **Chỉ cần tiền tố HOẶC hậu tố** (không cần cả hai).
3. **Đòi hỏi `words[i]` ngắn hơn hẳn `words[j]`** (loại trường hợp trùng độ dài).
4. **Count Prefix and Suffix Pairs II** (LeetCode 3045): `n` tới `10^5`, tổng độ dài tới `5 · 10^5`. Dùng Trie cặp ký tự như Cách 4.
5. **Prefix and Suffix Search** (LeetCode 745): tra từ theo tiền tố và hậu tố cho trước.

In [25]:
def list_pairs(words):
    n = len(words)
    return [(i, j) for i in range(n) for j in range(i + 1, n)
            if words[j].startswith(words[i]) and words[j].endswith(words[i])]


def count_prefix_or_suffix(words):
    n = len(words)
    return sum(words[j].startswith(words[i]) or words[j].endswith(words[i])
               for i in range(n) for j in range(i + 1, n))


print(list_pairs(["a", "aba", "ababa", "aa"]))        # [(0, 1), (0, 2), (0, 3), (1, 2)]
print(count_prefix_or_suffix(["a", "ab", "ba"]))      # (a,ab) tiền tố, (a,ba) hậu tố, (ab,ba) không -> 2
assert list_pairs(["a", "aba", "ababa", "aa"]) == [(0, 1), (0, 2), (0, 3), (1, 2)]
assert count_prefix_or_suffix(["a", "ab", "ba"]) == 2

[(0, 1), (0, 2), (0, 3), (1, 2)]
2


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. So từng ký tự | tự viết vòng kiểm tra tiền tố, hậu tố | `O(n² · L)` | `O(1)` | học cơ chế bên trong |
| 2. Slicing | `b[:k] == a and b[-k:] == a` | `O(n² · L)` | `O(L)` | khi thích nhìn rõ đoạn đầu/cuối |
| 3. `startswith`/`endswith` | hai hàm có sẵn | `O(n² · L)` | `O(1)` | **nên nộp** cho bài I |
| 4. Trie cặp ký tự | khóa `(đầu, cuối)` trong Trie | `O(n · L)` | `O(n · L)` | `n` rất lớn (bài II) |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `"banana".startswith("ban")`, `"banana".endswith("na")`, `"banana"[-3:]`, `"banana"[:0]`, `Solution().countPrefixSuffixPairs(["ab", "ab", "ab"])`.

**Bài 2 (tính tay).** Với `words = ["x", "xyx", "yx", "xyxyx"]`, liệt kê mọi cặp `i < j` thỏa đề và cho biết đáp án.

**Bài 3 (điền TODO).** Hoàn thành `count_pairs_todo(words)` theo Cách 3.

In [26]:
def count_pairs_todo(words):
    n = len(words)
    count = 0
    # TODO 1: vòng lặp i từ 0 tới n - 1
    #   TODO 2: vòng lặp j từ i + 1 tới n - 1
    #     TODO 3: nếu words[j] bắt đầu bằng words[i] VÀ kết thúc bằng words[i] thì count += 1
    return count

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `count_pairs_slicing2(words)` dùng **slicing** (Cách 2) và **không** dùng `startswith`/`endswith`. Phải xử lý đúng khi `words[i]` dài hơn `words[j]`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `longest_pair(words)` trả về cặp `(i, j)` (`i < j`) thỏa đề mà `len(words[i])` lớn nhất; nếu không có cặp nào trả `None`; hòa thì lấy cặp có `i` nhỏ nhất rồi `j` nhỏ nhất. (b) Viết `count_pairs_trie_check(words)` và đối chiếu với Cách 3 bằng test ngẫu nhiên.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `True`, `True`, `"ana"`, `""` (chuỗi rỗng), `3`. Bài 3: hai vòng `for` lồng nhau, điều kiện nối bằng `and`.
- **Mức 2 (vừa):** Bài 2: thử từng cặp: `"x"` là tiền tố/hậu tố của `"xyx"` và `"xyxyx"` nhưng không của `"yx"` (không bắt đầu bằng `x`); `"xyx"` thuộc `"xyxyx"`; `"yx"` không là tiền tố của `"xyxyx"`. Bài 4: dùng `len(a) <= len(b)` rồi `b[:len(a)] == a and b[-len(a):] == a`.
- **Mức 3 (sâu):** Bài 5(a): duyệt mọi cặp bằng `list_pairs`, rồi chọn bằng `max` với khóa `len(words[i])`, nhớ xử lý hòa bằng cách duyệt theo thứ tự và chỉ cập nhật khi **lớn hơn hẳn**. 5(b): dùng lại `count_pairs_trie` và so sánh trên input ngẫu nhiên có seed.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [27]:
# Bài 1
assert "banana".startswith("ban") is True
assert "banana".endswith("na") is True
assert "banana"[-3:] == "ana"
assert "banana"[:0] == ""
assert Solution().countPrefixSuffixPairs(["ab", "ab", "ab"]) == 3

# Bài 2
w2 = ["x", "xyx", "yx", "xyxyx"]
assert list_pairs(w2) == [(0, 1), (0, 3), (1, 3)]
assert Solution().countPrefixSuffixPairs(w2) == 3


# Bài 3
def count_pairs_todo_ans(words):
    n = len(words)
    count = 0
    for i in range(n):
        for j in range(i + 1, n):
            if words[j].startswith(words[i]) and words[j].endswith(words[i]):
                count += 1
    return count


for words, expected in examples:
    assert count_pairs_todo_ans(words) == expected


# Bài 4
def count_pairs_slicing2(words):
    count = 0
    for i in range(len(words)):
        a = words[i]
        for j in range(i + 1, len(words)):
            b = words[j]
            if len(a) <= len(b) and b[:len(a)] == a and b[-len(a):] == a:
                count += 1
    return count


for words, expected in examples:
    assert count_pairs_slicing2(words) == expected


# Bài 5(a)
def longest_pair(words):
    best = None
    for i, j in list_pairs(words):
        if best is None or len(words[i]) > len(words[best[0]]):
            best = (i, j)
    return best


assert longest_pair(["a", "aba", "ababa", "aa"]) == (1, 2)
assert longest_pair(["abab", "ab"]) is None
assert longest_pair(["a", "a", "a"]) == (0, 1)

# Bài 5(b)
rng = random.Random(7)
for _ in range(300):
    words = random_words(rng, "ab", 6, 20)
    assert count_pairs_trie(words) == count_pairs_startswith(words)
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đếm cặp `i < j` thỏa điều kiện trên hai chuỗi)

```python
from typing import List

class Solution:
    def countPrefixSuffixPairs(self, words: List[str]) -> int:
        n = len(words)
        count = 0
        for i in range(n):
            for j in range(i + 1, n):
                if words[j].startswith(words[i]) and words[j].endswith(words[i]):
                    count += 1
        return count
```

**Checklist**

- [ ] Chỉ đếm cặp `i < j` (`j` chạy từ `i + 1`)?
- [ ] `words[i]` là chuỗi **ngắn hơn hoặc bằng** đứng **trước**, `words[j]` là chuỗi đứng sau: dùng `words[j].startswith(words[i])`, không đảo ngược?
- [ ] Đủ **cả hai** điều kiện: tiền tố **và** hậu tố (`and`, không phải `or`)?
- [ ] Không dùng `in` (xuất hiện ở giữa không được tính)?
- [ ] Đã thử `["a", "a"]`, `["abab", "ab"]` và trường hợp một từ?
- [ ] Skeleton đúng `class Solution` / `def countPrefixSuffixPairs(self, words: List[str]) -> int`?